# FreshRoute: learn the cuOpt NIM API on Azure Container Apps

This notebook is a self-contained Python client for the NVIDIA cuOpt NIM deployed on
Azure Container Apps serverless GPU. It generates a synthetic Phoenix last-mile delivery
dataset, constructs the complete cuOpt routing payload, calls the NIM REST API directly,
validates the returned routes, and displays them with Azure Maps.

No demo web application, proxy API, or prebuilt solution is involved. The notebook solves
both the 120-order morning plan and a 144-order disruption recovery directly with cuOpt.
The exercise teaches API integration and constrained routing—not CPU/GPU benchmarking.


## 1. Configure direct service access

The cuOpt endpoint must be restricted to this laptop's public `/32`. Copy `.env.example`
to `.env`, replace the placeholders with the deployed Azure asset values, and run
`az login`. The notebook loads `.env` without placing its contents in notebook output.
Azure Maps uses the signed-in Azure CLI identity; no Maps key or bearer token is stored.

```bash
cp .env.example .env
az login
jupyter lab freshroute_cuopt_aca.ipynb
```


In [ ]:
import atexit
import math
import os
import random
import threading
import time
from collections import Counter
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlencode

import folium
import httpx
import msgpack
import pandas as pd
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display

# Work whether Jupyter starts in this sample directory or at the repository root.
env_candidates = [
    Path.cwd() / ".env",
    Path.cwd() / "inference" / "aca" / "cuopt" / ".env",
]
env_file = next((path for path in env_candidates if path.is_file()), None)
if env_file is None:
    raise RuntimeError("Create inference/aca/cuopt/.env from .env.example before running the notebook.")
load_dotenv(env_file, override=False)

CUOPT_BASE_URL = os.environ.get("CUOPT_BASE_URL", "").rstrip("/")
AZURE_MAPS_CLIENT_ID = os.environ.get("AZURE_MAPS_CLIENT_ID", "")
CUOPT_TIME_LIMIT_SECONDS = int(os.environ.get("CUOPT_TIME_LIMIT_SECONDS", "10"))
CUOPT_WAKE_TIMEOUT_SECONDS = int(os.environ.get("CUOPT_WAKE_TIMEOUT_SECONDS", "720"))

if not CUOPT_BASE_URL or "<" in CUOPT_BASE_URL:
    raise RuntimeError("Set CUOPT_BASE_URL in .env to the IP-restricted cuOpt NIM HTTPS URL.")
if not AZURE_MAPS_CLIENT_ID or "<" in AZURE_MAPS_CLIENT_ID:
    raise RuntimeError("Set AZURE_MAPS_CLIENT_ID in .env to the Azure Maps account client ID.")

print(f"Loaded configuration from {env_file}")
print(f"cuOpt NIM: {CUOPT_BASE_URL}")
print(f"Solver limit: {CUOPT_TIME_LIMIT_SECONDS} seconds per plan")


## 2. Generate the teaching dataset

A fixed random seed makes the scenario reproducible. Orders contain real geographic
coordinates around eight Phoenix-area service zones, plus weight, volume, service time,
customer time windows, priority, and cold-chain requirements. The fleet has three depots,
eight EV vans, four box trucks, and two refrigerated trucks.


In [ ]:
SEED = 20260803
ROUTE_COLORS = [
    "#76B900", "#00B4D8", "#F59E0B", "#A78BFA", "#FB7185", "#2DD4BF",
    "#60A5FA", "#F97316", "#C084FC", "#22C55E", "#EAB308", "#38BDF8",
    "#F472B6", "#84CC16",
]
DEPOTS = [
    {"id": "PHX", "name": "Phoenix Central", "latitude": 33.4484, "longitude": -112.0740},
    {"id": "TMP", "name": "Tempe East", "latitude": 33.4255, "longitude": -111.9400},
    {"id": "GLD", "name": "Glendale West", "latitude": 33.5387, "longitude": -112.1860},
]
CLUSTERS = [
    (33.4484, -112.0740, "Downtown Phoenix"),
    (33.4255, -111.9400, "Tempe"),
    (33.4942, -111.9261, "Scottsdale"),
    (33.4152, -111.8315, "Mesa"),
    (33.3062, -111.8413, "Chandler"),
    (33.5387, -112.1860, "Glendale"),
    (33.4353, -112.3577, "Goodyear"),
    (33.3528, -111.7890, "Gilbert"),
]


def vehicle(vehicle_id: int, depot_index: int, kind: str, refrigerated: bool = False) -> dict:
    if kind == "EV Van":
        weight, volume, max_km, cost_factor, time_factor = 900, 60, 210, 1.0, 1.0
    elif kind == "Refrigerated Truck":
        weight, volume, max_km, cost_factor, time_factor = 2200, 120, 300, 1.22, 1.12
    else:
        weight, volume, max_km, cost_factor, time_factor = 2500, 140, 320, 1.16, 1.08
    return {
        "id": f"FR-{vehicle_id:02d}", "type": kind,
        "type_index": 0 if kind == "EV Van" else 1,
        "depot_index": depot_index, "depot_id": DEPOTS[depot_index]["id"],
        "capacity_weight": weight, "capacity_volume": volume,
        "refrigerated": refrigerated, "shift_start": 480, "shift_end": 1260,
        "break_start": 720, "break_end": 810, "break_duration": 45,
        "max_route_minutes": 780, "max_distance_km": max_km,
        "cost_factor": cost_factor, "time_factor": time_factor,
        "available": True, "color": ROUTE_COLORS[vehicle_id - 1],
    }


VEHICLES = [
    *[vehicle(index + 1, index % 3, "EV Van") for index in range(8)],
    *[vehicle(index + 9, (index + 1) % 3, "Box Truck") for index in range(4)],
    vehicle(13, 0, "Refrigerated Truck", True),
    vehicle(14, 1, "Refrigerated Truck", True),
]


def generate_orders() -> list[dict]:
    rng = random.Random(SEED)
    orders = []
    for index in range(144):
        latitude, longitude, zone = CLUSTERS[index % len(CLUSTERS)]
        latitude += rng.gauss(0, 0.018)
        longitude += rng.gauss(0, 0.022)
        window_start = [540, 600, 660, 720, 780][index % 5]
        orders.append({
            "id": f"ORD-{index + 1:04d}", "order_index": index,
            "latitude": round(latitude, 6), "longitude": round(longitude, 6),
            "zone": zone, "demand_weight": rng.randint(24, 62),
            "demand_volume": rng.randint(2, 5), "service_minutes": rng.randint(5, 9),
            "window_start": window_start, "window_end": min(window_start + 720, 1260),
            "cold_chain": index % 7 == 0, "priority": index >= 120,
        })
    return orders


ALL_ORDERS = generate_orders()
morning_orders = ALL_ORDERS[:120]
disrupted_orders = ALL_ORDERS
morning_vehicles = [dict(item) for item in VEHICLES]
disrupted_vehicles = [dict(item) for item in VEHICLES]
disrupted_vehicles[2]["available"] = False

display(pd.DataFrame([{
    "depots": len(DEPOTS), "morning_orders": len(morning_orders),
    "priority_orders_added": len(disrupted_orders) - len(morning_orders),
    "recovery_orders": len(disrupted_orders), "original_drivers": len(VEHICLES),
    "available_after_callout": sum(item["available"] for item in disrupted_vehicles),
}]))


In [ ]:
orders_df = pd.DataFrame(disrupted_orders)
fleet_df = pd.DataFrame(disrupted_vehicles)
display(Markdown("### Orders by service zone"))
display(orders_df.groupby("zone").agg(
    orders=("id", "count"), priority=("priority", "sum"), cold_chain=("cold_chain", "sum")
))
display(Markdown("### Fleet after the driver callout"))
display(fleet_df.groupby(["type", "available"]).agg(vehicles=("id", "count")))


## 3. Construct the cuOpt payload

cuOpt receives cost and travel-time matrices plus task and fleet constraints. This notebook
constructs both the morning and disrupted payloads in Python. The disrupted travel-time
matrices apply a congestion multiplier to central or cross-city movements.


In [ ]:
def haversine_km(left: tuple[float, float], right: tuple[float, float]) -> float:
    lat1, lon1 = map(math.radians, left)
    lat2, lon2 = map(math.radians, right)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * math.asin(math.sqrt(value))


def matrix(nodes: list[dict], type_factor: float, congestion: bool, travel: bool) -> list[list[float]]:
    result = []
    for left in nodes:
        row = []
        for right in nodes:
            distance = haversine_km(
                (left["latitude"], left["longitude"]),
                (right["latitude"], right["longitude"]),
            )
            if travel:
                value = distance / 34.0 * 60.0 * type_factor
                central = left.get("zone") == "Downtown Phoenix" or right.get("zone") == "Downtown Phoenix"
                cross_core = (left["longitude"] < -112.02) != (right["longitude"] < -112.02)
                if congestion and (central or cross_core):
                    value *= 1.55
            else:
                value = distance * type_factor
            row.append(round(value, 4))
        result.append(row)
    return result


def build_payload(orders: list[dict], vehicles: list[dict], congestion: bool) -> dict:
    nodes = [*DEPOTS, *orders]
    available = [item for item in vehicles if item["available"]]
    cold_vehicle_indexes = [index for index, item in enumerate(available) if item["refrigerated"]]
    return {
        "cost_matrix_data": {"data": {
            "0": matrix(nodes, 1.0, False, False),
            "1": matrix(nodes, 1.16, False, False),
        }},
        "travel_time_matrix_data": {"data": {
            "0": matrix(nodes, 1.0, congestion, True),
            "1": matrix(nodes, 1.1, congestion, True),
        }},
        "task_data": {
            "task_locations": list(range(len(DEPOTS), len(nodes))),
            "task_ids": [item["id"] for item in orders],
            "demand": [
                [item["demand_weight"] for item in orders],
                [item["demand_volume"] for item in orders],
            ],
            "task_time_windows": [[item["window_start"], item["window_end"]] for item in orders],
            "service_times": [item["service_minutes"] for item in orders],
            "order_vehicle_match": [
                {"order_id": item["order_index"], "vehicle_ids": cold_vehicle_indexes}
                for item in orders if item["cold_chain"]
            ],
        },
        "fleet_data": {
            "vehicle_locations": [[item["depot_index"], item["depot_index"]] for item in available],
            "vehicle_ids": [item["id"] for item in available],
            "capacities": [
                [item["capacity_weight"] for item in available],
                [item["capacity_volume"] for item in available],
            ],
            "vehicle_time_windows": [[item["shift_start"], item["shift_end"]] for item in available],
            "vehicle_break_time_windows": [[
                [item["break_start"], item["break_end"]] for item in available
            ]],
            "vehicle_break_durations": [[item["break_duration"] for item in available]],
            "vehicle_max_times": [item["max_route_minutes"] for item in available],
            "vehicle_max_costs": [int(item["max_distance_km"]) for item in available],
            "vehicle_types": [item["type_index"] for item in available],
        },
        "solver_config": {"time_limit": CUOPT_TIME_LIMIT_SECONDS},
    }


morning_payload = build_payload(morning_orders, morning_vehicles, congestion=False)
disrupted_payload = build_payload(disrupted_orders, disrupted_vehicles, congestion=True)

display(pd.DataFrame([
    {
        "plan": "Morning", "matrix_locations": len(morning_payload["cost_matrix_data"]["data"]["0"]),
        "tasks": len(morning_payload["task_data"]["task_ids"]),
        "vehicles": len(morning_payload["fleet_data"]["vehicle_ids"]),
    },
    {
        "plan": "Disrupted", "matrix_locations": len(disrupted_payload["cost_matrix_data"]["data"]["0"]),
        "tasks": len(disrupted_payload["task_data"]["task_ids"]),
        "vehicles": len(disrupted_payload["fleet_data"]["vehicle_ids"]),
    },
]).set_index("plan"))
display(pd.DataFrame([row[:5] for row in disrupted_payload["cost_matrix_data"]["data"]["0"][:5]]))


## 4. Implement the direct cuOpt NIM client

The NIM API is asynchronous: wait for health, submit the routing payload, poll the request,
then retrieve the solution. JSON and MessagePack responses are both supported. Submission
is never automatically retried, avoiding duplicate optimization jobs.


In [ ]:
def decode_response(response: httpx.Response):
    content_type = response.headers.get("content-type", "").lower()
    if "msgpack" in content_type or "octet-stream" in content_type:
        return msgpack.unpackb(response.content, raw=False)
    try:
        return response.json()
    except (UnicodeDecodeError, ValueError):
        return msgpack.unpackb(response.content, raw=False)


def wait_for_cuopt(client: httpx.Client) -> float:
    started = time.perf_counter()
    deadline = time.monotonic() + CUOPT_WAKE_TIMEOUT_SECONDS
    last_error = None
    while time.monotonic() < deadline:
        try:
            response = client.get(f"{CUOPT_BASE_URL}/cuopt/health", timeout=30)
            if response.status_code == 200:
                return time.perf_counter() - started
            last_error = f"HTTP {response.status_code}"
        except httpx.HTTPError as exc:
            last_error = str(exc)
        time.sleep(2)
    raise TimeoutError(f"cuOpt did not become ready: {last_error}")


def solve_with_cuopt(problem: dict, plan_name: str) -> tuple[dict, dict]:
    with httpx.Client(headers={"Accept": "application/json"}) as client:
        wake_seconds = wait_for_cuopt(client)
        started = time.perf_counter()
        response = client.post(
            f"{CUOPT_BASE_URL}/cuopt/request", json=problem,
            headers={"Accept": "application/json", "Content-Type": "application/json"}, timeout=60,
        )
        response.raise_for_status()
        request_body = decode_response(response)
        request_id = request_body.get("reqId")
        if not request_id:
            raise RuntimeError(f"cuOpt did not return a request ID: {request_body}")

        deadline = time.monotonic() + CUOPT_TIME_LIMIT_SECONDS + 120
        while time.monotonic() < deadline:
            status_response = client.get(f"{CUOPT_BASE_URL}/cuopt/request/{request_id}", timeout=30)
            status_response.raise_for_status()
            status_body = decode_response(status_response)
            request_status = status_body.get("status") if isinstance(status_body, dict) else status_body
            if request_status == "completed":
                break
            if request_status in {"failed", "error"}:
                raise RuntimeError(f"cuOpt job failed: {status_body}")
            time.sleep(1)
        else:
            raise TimeoutError(f"cuOpt request {request_id} exceeded the polling deadline.")

        solution_response = client.get(f"{CUOPT_BASE_URL}/cuopt/solution/{request_id}", timeout=60)
        solution_response.raise_for_status()
        solution = decode_response(solution_response)
    return solution, {
        "plan": plan_name, "request_id": request_id,
        "gpu_wake_seconds": round(wake_seconds, 3),
        "request_to_solution_seconds": round(time.perf_counter() - started, 3),
    }


## 5. Convert and validate cuOpt vehicle data

The validator below is independent of cuOpt. It verifies complete, duplicate-free task
coverage plus time windows, two capacity dimensions, cold-chain compatibility, driver
breaks, route time, route distance, and vehicle availability.


In [ ]:
def parse_and_validate(solution: dict, orders: list[dict], vehicles: list[dict], problem: dict):
    solver_response = solution.get("response", {}).get("solver_response", {})
    if solver_response.get("status") != 0:
        raise RuntimeError(solver_response.get("msg") or f"Unexpected response: {solver_response}")

    order_by_id = {item["id"]: item for item in orders}
    vehicle_by_id = {item["id"]: item for item in vehicles}
    fleet = problem["fleet_data"]
    fleet_index = {vehicle_id: index for index, vehicle_id in enumerate(fleet["vehicle_ids"])}
    matrices = problem["cost_matrix_data"]["data"]
    routes, stop_rows, route_rows = [], [], []
    violations = {
        "unknown_vehicles": [], "unavailable_vehicles": [], "late_or_missing_arrivals": [],
        "capacity_violations": [], "compatibility_violations": [], "break_violations": [],
        "route_time_violations": [], "route_distance_violations": [],
    }

    for vehicle_id, vehicle_solution in solver_response.get("vehicle_data", {}).items():
        if vehicle_id not in vehicle_by_id or vehicle_id not in fleet_index:
            violations["unknown_vehicles"].append(vehicle_id)
            continue
        item = vehicle_by_id[vehicle_id]
        if not item["available"]:
            violations["unavailable_vehicles"].append(vehicle_id)
        types = vehicle_solution.get("type", [])
        task_ids = vehicle_solution.get("task_id", [])
        arrivals = vehicle_solution.get("arrival_stamp", [])
        delivery_ids = [
            str(task_id) for task_id, task_type in zip(task_ids, types)
            if task_type == "Delivery" and str(task_id) in order_by_id
        ]
        if not delivery_ids:
            continue
        arrival_by_id = {
            str(task_id): int(round(arrival))
            for task_id, task_type, arrival in zip(task_ids, types, arrivals)
            if task_type == "Delivery" and str(task_id) in order_by_id
        }
        route_nodes = vehicle_solution.get("route") or [
            item["depot_index"],
            *[order_by_id[order_id]["order_index"] + len(DEPOTS) for order_id in delivery_ids],
            item["depot_index"],
        ]
        position = fleet_index[vehicle_id]
        cost_matrix = matrices[str(fleet["vehicle_types"][position])]
        distance = sum(cost_matrix[left][right] for left, right in zip(route_nodes, route_nodes[1:]))
        weight = sum(order_by_id[order_id]["demand_weight"] for order_id in delivery_ids)
        volume = sum(order_by_id[order_id]["demand_volume"] for order_id in delivery_ids)
        depot_arrivals = [float(value) for kind, value in zip(types, arrivals) if kind == "Depot"]
        elapsed = depot_arrivals[-1] - item["shift_start"] if depot_arrivals else 0

        if weight > item["capacity_weight"] or volume > item["capacity_volume"]:
            violations["capacity_violations"].append(vehicle_id)
        if elapsed >= 240 and "Break" not in types:
            violations["break_violations"].append(vehicle_id)
        if elapsed > item["max_route_minutes"]:
            violations["route_time_violations"].append(vehicle_id)
        if distance > item["max_distance_km"]:
            violations["route_distance_violations"].append(vehicle_id)

        stops = []
        for sequence, order_id in enumerate(delivery_ids, 1):
            order = order_by_id[order_id]
            arrival = arrival_by_id.get(order_id)
            if arrival is None or not (order["window_start"] <= arrival <= order["window_end"]):
                violations["late_or_missing_arrivals"].append(order_id)
            if order["cold_chain"] and not item["refrigerated"]:
                violations["compatibility_violations"].append(order_id)
            stop = {
                "order_id": order_id, "latitude": order["latitude"], "longitude": order["longitude"],
                "arrival_minute": arrival, "window_start": order["window_start"],
                "window_end": order["window_end"], "cold_chain": order["cold_chain"],
                "priority": order["priority"],
            }
            stops.append(stop)
            stop_rows.append({"vehicle_id": vehicle_id, "sequence": sequence, **stop})

        routes.append({
            "vehicle_id": vehicle_id, "vehicle_type": item["type"], "depot_id": item["depot_id"],
            "color": item["color"], "stops": stops, "total_distance_km": round(distance, 2),
            "total_travel_minutes": round(elapsed, 1),
        })
        route_rows.append({
            "vehicle_id": vehicle_id, "vehicle_type": item["type"], "stops": len(stops),
            "distance_km": round(distance, 2), "route_minutes": round(elapsed, 1),
            "weight_utilization": round(weight / item["capacity_weight"], 3),
            "volume_utilization": round(volume / item["capacity_volume"], 3),
            "break_scheduled": "Break" in types,
        })

    seen = [row["order_id"] for row in stop_rows]
    counts = Counter(seen)
    validation = {
        "missing_orders": sorted(set(order_by_id) - set(seen)),
        "duplicate_orders": sorted(order_id for order_id, count in counts.items() if count > 1),
        **{key: sorted(set(value)) for key, value in violations.items()},
    }
    validation["valid"] = all(not value for key, value in validation.items() if key != "valid")
    return pd.DataFrame(route_rows), pd.DataFrame(stop_rows), validation, routes


## 6. Solve the 120-order morning plan directly

This is the baseline dispatch plan. Its request ID and timings come from the cuOpt NIM,
not from a recorded fixture or application wrapper.


In [ ]:
morning_solution, morning_timing = solve_with_cuopt(morning_payload, "Morning")
morning_routes_df, morning_stops_df, morning_validation, morning_routes = parse_and_validate(
    morning_solution, morning_orders, morning_vehicles, morning_payload
)
assert len(morning_stops_df) == 120
assert morning_validation["valid"], morning_validation
display(pd.DataFrame([morning_timing]))
display(morning_routes_df.sort_values("vehicle_id"))
display(morning_validation)


## 7. Visualize the plan with Azure Maps

`AzureCliCredential` obtains short-lived Azure Maps tokens. Route Directions converts each
cuOpt stop sequence into road-following geometry. Static Render produces a fixed zoom-10
overview, while Folium provides an interactive Azure Maps view with pan, wheel zoom, route
layer toggles, and stop tooltips. A loopback-only tile proxy keeps tokens in Python memory
and Azure request headers; the notebook output contains no token or Maps key.


In [ ]:
AZURE_MAPS_RENDER_URL = "https://atlas.microsoft.com/map/static"
AZURE_MAPS_TILE_URL = "https://atlas.microsoft.com/map/tile"
AZURE_MAPS_DIRECTIONS_URL = "https://atlas.microsoft.com/route/directions"
PHOENIX_CENTER = (33.43, -112.073)
PHOENIX_OVERVIEW_ZOOM = 10
maps_credential = AzureCliCredential()


def maps_headers(accept: str = "application/json") -> dict[str, str]:
    token = maps_credential.get_token("https://atlas.microsoft.com/.default")
    return {
        "Authorization": f"Bearer {token.token}",
        "x-ms-client-id": AZURE_MAPS_CLIENT_ID,
        "Accept": accept,
    }


def add_road_geometry(routes: list[dict]) -> None:
    """Attach Azure Maps road geometry without changing cuOpt's stop order."""
    for route in routes:
        depot = next(item for item in DEPOTS if item["id"] == route["depot_id"])
        waypoints = [depot, *route["stops"], depot]
        body = {
            "type": "FeatureCollection",
            "features": [
                {
                    "type": "Feature",
                    "geometry": {
                        "type": "Point",
                        "coordinates": [item["longitude"], item["latitude"]],
                    },
                    "properties": {"pointIndex": index, "pointType": "waypoint"},
                }
                for index, item in enumerate(waypoints)
            ],
            "optimizeRoute": "fastestWithoutTraffic",
            "routeOutputOptions": ["routePath"],
            "travelMode": "truck" if "Truck" in route["vehicle_type"] else "driving",
        }
        response = httpx.post(
            AZURE_MAPS_DIRECTIONS_URL, params={"api-version": "2025-01-01"},
            headers={**maps_headers("application/geo+json"), "Content-Type": "application/geo+json"},
            json=body, timeout=90,
        )
        response.raise_for_status()
        feature = next(
            item for item in response.json()["features"]
            if item.get("geometry", {}).get("type") == "MultiLineString"
        )
        route["road_geometry"] = [
            [latitude, longitude]
            for line in feature["geometry"]["coordinates"]
            for longitude, latitude in line
        ]


def retry_map_get(params: list[tuple[str, str]], attempts: int = 4) -> httpx.Response:
    for attempt in range(1, attempts + 1):
        try:
            response = httpx.get(
                AZURE_MAPS_RENDER_URL, params=params, headers=maps_headers("image/png"),
                timeout=httpx.Timeout(90, connect=90),
            )
            response.raise_for_status()
            return response
        except httpx.HTTPError:
            if attempt == attempts:
                raise
            time.sleep(min(2 ** (attempt - 1), 8))
    raise RuntimeError("Azure Maps request did not return a response.")


def coordinate(item: dict) -> str:
    return f"{item['longitude']} {item['latitude']}"


def sample_geometry(points: list[list[float]], maximum: int = 28) -> list[list[float]]:
    if len(points) <= maximum:
        return points
    step = (len(points) - 1) / (maximum - 1)
    return [points[round(index * step)] for index in range(maximum)]


def route_path(route: dict) -> str:
    if route.get("road_geometry"):
        points = [f"{longitude} {latitude}" for latitude, longitude in sample_geometry(route["road_geometry"])]
    else:
        depot = next(item for item in DEPOTS if item["id"] == route["depot_id"])
        points = [coordinate(depot), *[coordinate(stop) for stop in route["stops"]], coordinate(depot)]
    return f"lc{route['color'].lstrip('#')}|lw4||" + "|".join(points)


def render_map(paths: list[str], pin_groups: list[tuple[str, list[dict]]], bounds: list[dict], title: str) -> bytes:
    params = [
        ("api-version", "2024-04-01"), ("tilesetId", "microsoft.base.road"),
        ("center", f"{PHOENIX_CENTER[1]},{PHOENIX_CENTER[0]}"),
        ("zoom", str(PHOENIX_OVERVIEW_ZOOM)), ("width", "1200"), ("height", "700"),
    ]
    params.extend(("path", value) for value in paths)
    for color, items in pin_groups:
        for start in range(0, len(items), 50):
            points = "|".join(coordinate(item) for item in items[start:start + 50])
            params.append(("pins", f"default|sc0.45|co{color.lstrip('#')}||{points}"))
    if len(AZURE_MAPS_RENDER_URL) + len(urlencode(params)) > 7800:
        raise ValueError("Azure Maps request is too large; reduce the number of routes per panel.")
    response = retry_map_get(params)
    if not response.headers.get("content-type", "").startswith("image/"):
        raise RuntimeError("Azure Maps did not return an image.")
    display(Markdown(f"### {title} — fixed overview"))
    display(Image(data=response.content))
    return response.content


def render_route_panels(routes: list[dict], title: str, extra_pins=None) -> list[bytes]:
    panels = [routes[start:start + 8] for start in range(0, len(routes), 8)]
    images = []
    for index, panel in enumerate(panels, 1):
        panel_title = title if len(panels) == 1 else f"{title} — panel {index} of {len(panels)}"
        bounds = [*DEPOTS, *[stop for route in panel for stop in route["stops"]]]
        images.append(render_map(
            [route_path(route) for route in panel],
            [("76B900", DEPOTS), *(extra_pins or [])], bounds, panel_title,
        ))
    return images


_tile_cache: dict[tuple[int, int, int], bytes] = {}


class AzureMapsTileHandler(BaseHTTPRequestHandler):
    def log_message(self, *_args) -> None:
        pass

    def do_GET(self) -> None:
        try:
            _, zoom, x, y_png = self.path.split("?", 1)[0].split("/")
            key = int(zoom), int(x), int(y_png.removesuffix(".png"))
        except (ValueError, TypeError):
            self.send_error(404)
            return
        if key not in _tile_cache:
            response = httpx.get(
                AZURE_MAPS_TILE_URL,
                params={
                    "api-version": "2024-04-01", "tilesetId": "microsoft.base.road",
                    "zoom": key[0], "x": key[1], "y": key[2], "tileSize": 256,
                },
                headers=maps_headers("image/png"), timeout=60,
            )
            if response.status_code != 200:
                self.send_error(response.status_code)
                return
            _tile_cache[key] = response.content
        content = _tile_cache[key]
        self.send_response(200)
        self.send_header("Content-Type", "image/png")
        self.send_header("Content-Length", str(len(content)))
        self.send_header("Cache-Control", "private, max-age=300")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(content)


try:
    azure_maps_tile_server.shutdown()
except NameError:
    pass
azure_maps_tile_server = ThreadingHTTPServer(("127.0.0.1", 0), AzureMapsTileHandler)
threading.Thread(target=azure_maps_tile_server.serve_forever, daemon=True).start()
atexit.register(azure_maps_tile_server.shutdown)
AZURE_MAPS_TILE_TEMPLATE = (
    f"http://127.0.0.1:{azure_maps_tile_server.server_port}/{{z}}/{{x}}/{{y}}.png"
)


def minute_label(value: int | None) -> str:
    return "n/a" if value is None else f"{value // 60:02d}:{value % 60:02d}"


def interactive_route_map(
    routes: list[dict], title: str, priority=None, unavailable=None,
) -> folium.Map:
    route_map = folium.Map(
        location=PHOENIX_CENTER, zoom_start=PHOENIX_OVERVIEW_ZOOM, tiles=None,
        control_scale=True, prefer_canvas=True, width="100%", height=620,
    )
    folium.TileLayer(
        tiles=AZURE_MAPS_TILE_TEMPLATE, attr="Microsoft Azure Maps and TomTom",
        name="Azure Maps roads", overlay=False, control=False, max_zoom=18,
    ).add_to(route_map)

    depots = folium.FeatureGroup(name="Depots", show=True).add_to(route_map)
    for depot in DEPOTS:
        folium.CircleMarker(
            [depot["latitude"], depot["longitude"]], radius=7, color="#315700",
            weight=2, fill=True, fill_color="#76B900", fill_opacity=1,
            tooltip=f"{depot['id']} · {depot['name']}",
        ).add_to(depots)

    for route in routes:
        group = folium.FeatureGroup(
            name=f"{route['vehicle_id']} · {route['vehicle_type']} · {len(route['stops'])} stops",
            show=True,
        ).add_to(route_map)
        depot = next(item for item in DEPOTS if item["id"] == route["depot_id"])
        geometry = route.get("road_geometry") or [
            [depot["latitude"], depot["longitude"]],
            *[[stop["latitude"], stop["longitude"]] for stop in route["stops"]],
            [depot["latitude"], depot["longitude"]],
        ]
        folium.PolyLine(
            geometry, color=route["color"], weight=5, opacity=0.86,
            tooltip=f"{route['vehicle_id']} · {len(route['stops'])} stops",
        ).add_to(group)
        for sequence, stop in enumerate(route["stops"], 1):
            radius = 5 if stop["priority"] else 3
            fill = "#F59E0B" if stop["priority"] else route["color"]
            folium.CircleMarker(
                [stop["latitude"], stop["longitude"]], radius=radius,
                color=route["color"], weight=1, fill=True, fill_color=fill, fill_opacity=0.92,
                tooltip=(
                    f"{route['vehicle_id']} stop {sequence}: {stop['order_id']} · "
                    f"arrival {minute_label(stop['arrival_minute'])}"
                ),
            ).add_to(group)

    for label, items, color in [
        ("Priority orders", priority or [], "#F59E0B"),
        ("Unavailable-driver stops", unavailable or [], "#EF4444"),
    ]:
        if not items:
            continue
        group = folium.FeatureGroup(name=f"{label} · {len(items)}", show=True).add_to(route_map)
        for item in items:
            folium.CircleMarker(
                [item["latitude"], item["longitude"]], radius=5, color=color,
                weight=2, fill=True, fill_color=color, fill_opacity=0.9, tooltip=item["id"],
            ).add_to(group)

    folium.LayerControl(collapsed=False, position="topright").add_to(route_map)
    display(Markdown(f"### {title} — interactive"))
    display(route_map)
    return route_map


add_road_geometry(morning_routes)
morning_map_images = render_route_panels(morning_routes, "cuOpt morning plan: 120 orders")
assert all(image.startswith(b"\x89PNG") for image in morning_map_images)
morning_interactive_map = interactive_route_map(morning_routes, "cuOpt morning plan: 120 orders")


## 8. Apply the 2:15 PM disruption

Twenty-four priority orders arrive, driver `FR-03` calls out, and the travel-time matrices
now reflect congestion. Orange pins are the new orders; red pins are stops that were on the
unavailable driver's morning route.


In [ ]:
priority_orders = [item for item in disrupted_orders if item["priority"]]
unavailable_ids = set(morning_stops_df.loc[morning_stops_df["vehicle_id"] == "FR-03", "order_id"])
unavailable_orders = [item for item in morning_orders if item["id"] in unavailable_ids]
impact = pd.DataFrame([{
    "event_time": "2:15 PM", "priority_orders_added": len(priority_orders),
    "unavailable_driver": "FR-03", "orders_on_unavailable_route": len(unavailable_orders),
    "orders_requiring_replanning": len(priority_orders) + len(unavailable_orders),
}])
display(impact)
disruption_map = render_map(
    [], [("76B900", DEPOTS), ("F59E0B", priority_orders), ("EF4444", unavailable_orders)],
    [*DEPOTS, *priority_orders, *unavailable_orders], "2:15 PM disruption",
)
assert disruption_map.startswith(b"\x89PNG")
disruption_interactive_map = interactive_route_map(
    [], "2:15 PM disruption", priority=priority_orders, unavailable=unavailable_orders
)


## 9. Re-optimize all 144 orders directly with cuOpt

This is a new cuOpt NIM request using the disrupted payload: 144 tasks, 13 available
vehicles, congested travel times, and the full constraint set.


In [ ]:
recovery_solution, recovery_timing = solve_with_cuopt(disrupted_payload, "Disruption recovery")
recovery_routes_df, recovery_stops_df, recovery_validation, recovery_routes = parse_and_validate(
    recovery_solution, disrupted_orders, disrupted_vehicles, disrupted_payload
)
assert len(recovery_stops_df) == 144
assert recovery_validation["valid"], recovery_validation
display(pd.DataFrame([recovery_timing]))
display(recovery_routes_df.sort_values("vehicle_id"))
display(recovery_stops_df.head(12))
display(recovery_validation)


In [ ]:
add_road_geometry(recovery_routes)
recovery_map_images = render_route_panels(
    recovery_routes, "cuOpt recovered plan: 144 orders", [("F59E0B", priority_orders)]
)
assert all(image.startswith(b"\x89PNG") for image in recovery_map_images)
recovery_interactive_map = interactive_route_map(
    recovery_routes, "cuOpt recovered plan: 144 orders"
)

# Prove that the loopback tile bridge works and that rendered HTML contains no credential.
tile_probe = httpx.get(AZURE_MAPS_TILE_TEMPLATE.format(z=10, x=193, y=410), timeout=60)
tile_probe.raise_for_status()
assert tile_probe.headers["content-type"] == "image/png"
assert all(len(route["road_geometry"]) > len(route["stops"]) for route in recovery_routes)
interactive_html = recovery_interactive_map.get_root().render()
assert "Bearer " not in interactive_html and "subscription-key" not in interactive_html

outcome = pd.DataFrame([
    {
        "plan": "Morning", "orders": len(morning_stops_df), "drivers_used": len(morning_routes_df),
        "dropped_orders": len(morning_validation["missing_orders"]),
        "on_time_percentage": 100.0,
    },
    {
        "plan": "Recovered", "orders": len(recovery_stops_df), "drivers_used": len(recovery_routes_df),
        "dropped_orders": len(recovery_validation["missing_orders"]),
        "on_time_percentage": 100.0,
    },
]).set_index("plan")
display(outcome)
display(pd.DataFrame([morning_timing, recovery_timing]).set_index("plan"))


## What developers should take away

1. Convert operational data into matrices, task constraints, and fleet constraints.
2. Submit the payload to `/cuopt/request` and retain the returned request ID.
3. Poll `/cuopt/request/{request_id}` and retrieve `/cuopt/solution/{request_id}`.
4. Convert `vehicle_data` into dispatch records and independently validate the result.
5. Keep the cuOpt NIM behind a trusted, authenticated application boundary in production.

SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.  
SPDX-License-Identifier: Apache-2.0
